# Image sharpness vs. flight state (rekon10)

**Question.** The in-flight OAK-D color stills look mostly unusable (arcs, radial streaks, rolling-shutter bands), while some at-rest frames are sharp. Is that **motion blur** (long exposure x airframe motion), **motor vibration**, or **focus/VCM hunting**? Derive a focus metric, apply it to every still, and correlate it with **exposure**, **airframe motion** (OAK-D IMU from the `.feat`), **motor state** (FC `RCOU` / arm-disarm), **EKF velocity** (`XKF1`) and **vibration** (`VIBE`) -- specifically: are there *sharp* frames while the motors were live, or *blurry* ones at rest?

**Note on the mono images.** These stills are the **12 MP color** camera. The features (VINS) run on the **400p rectified mono** stereo pair, which is *not* saved to disk (consumed in-pipeline: mono -> depth -> rectified -> feature tracker; only disparity `.png` and color `.jpg` are written). Mono would be a far better SfM/quality target (global shutter, short exposure, no VCM) -- capturing it is a small tracker change.

Audit-trail conventions (same as the VIO comparison notebook): provenance + input hashes, structured dicts, inline figures, machine-readable summary in `derived/`.

In [ ]:
# --- papermill parameters ---
flight_dir = "/home/jovyan/datasets/flights/rekon10/260712-crash"

In [ ]:
import sys, os, glob, json, hashlib, subprocess, struct, collections
sys.path.insert(0, "/home/jovyan/coordinator/analysis")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import laplace, sobel
from ardupilot_log import parse_log
from vio_ekf_compare import align_time

D = flight_dir
derived = f"{D}/derived"; os.makedirs(derived, exist_ok=True)
feat    = sorted(glob.glob(f"{D}/**/*.feat", recursive=True))[0]
fc_bin  = sorted(glob.glob(f"{D}/*.bin"))[0]
cap_dir = os.path.dirname(feat)
stills  = sorted(p for p in glob.glob(f"{cap_dir}/*.jpg") if os.path.getsize(p) > 0)
def sha256(p):
    h=hashlib.sha256(); h.update(open(p,"rb").read()); return h.hexdigest()
try: git_sha = subprocess.check_output(["git","-C","/home/jovyan/coordinator","rev-parse","HEAD"], text=True).strip()
except Exception: git_sha = "unknown"
zero = {ext: len([p for p in glob.glob(f"{cap_dir}/*.{ext}") if os.path.getsize(p)==0]) for ext in ("jpg","png","json")}
print(f"usable stills: {len(stills)}  |  zero-size in session: {zero} (total {sum(zero.values())})")

## Focus metric

**Variance of the Laplacian** -- the standard no-reference focus measure: a sharp frame has strong second-derivative (edge) response, so its Laplacian has high variance; blur kills high frequencies and the variance collapses. **Tenengrad** (mean squared Sobel gradient) as a cross-check. Grayscale, resized to a fixed long side so the metric is comparable across frames.

In [ ]:
def focus_metrics(path, long_side=1024):
    im = Image.open(path).convert("L")
    w, h = im.size
    s = long_side / max(w, h)
    if s < 1.0:
        im = im.resize((max(1, int(w*s)), max(1, int(h*s))))
    a = np.asarray(im, dtype=np.float64)
    gx, gy = sobel(a, axis=0), sobel(a, axis=1)
    return dict(var_laplacian=float(laplace(a).var()),
                tenengrad=float((gx*gx + gy*gy).mean()), brightness=float(a.mean()))

In [ ]:
rows = []
for p in stills:
    sc = json.load(open(p[:-4] + ".json")) if os.path.exists(p[:-4] + ".json") else {}
    rows.append(dict(name=os.path.basename(p),
                     monotonic_s=sc.get("monotonic_ns", np.nan)/1e9,
                     wall_unix=sc.get("wall_clock_unix", np.nan),
                     exposure_ms=sc.get("exposure_us", np.nan)/1000.0,
                     iso=sc.get("iso", np.nan), **focus_metrics(p)))
S = pd.DataFrame(rows).sort_values("monotonic_s").reset_index(drop=True)
print(f"{len(S)} stills scored. var_laplacian [{S.var_laplacian.min():.0f}, {S.var_laplacian.max():.0f}]")
print(f"exposure_ms: min {S.exposure_ms.min():.1f} median {S.exposure_ms.median():.1f} max {S.exposure_ms.max():.1f}")
display(S[["name","var_laplacian","tenengrad","exposure_ms","iso"]].head())

## Place each still on the flight timeline, and tag motor / velocity / vibration

The OAK-D IMU (`.feat`) and the still sidecars share the tracker's monotonic clock, so a still maps directly onto the `.feat` timeline; motion onset is read from the OAK-D gyro. The FC clock is offset (no RTC/NTP) -- recover it by cross-correlating OAK-D gyro vs FC `IMU` gyro, then tag each still with FC arm/disarm, motor PWM (`RCOU`), EKF speed (`XKF1` VN/VE/VD) and vibration (`VIBE`).

In [ ]:
HDR = struct.Struct("<ddHI")
data = open(feat, "rb").read(); n=len(data); off=0; t0=None; tm=[]; gyro=[]
while off < n:
    if n-off < HDR.size: break
    t_mono,t_unix,sid,length = HDR.unpack_from(data, off)
    if n-off < HDR.size+length: break
    if sid==0 and length==56:
        d = struct.unpack("<7d", data[off+HDR.size:off+HDR.size+length])
        if t0 is None: t0=t_mono
        tm.append(t_mono-t0); gyro.append((d[4],d[5],d[6]))
    off += HDR.size+length
tm=np.array(tm); gmag=np.linalg.norm(gyro,axis=1)
S["t_rel"] = S["monotonic_s"] - t0
B=2.0; buck=collections.defaultdict(list)
for t,g in zip(tm,gmag): buck[int(t//B)].append(g)
grms={k*B: np.sqrt(np.mean(np.square(v))) for k,v in buck.items()}
base=min([grms[k] for k in grms if k<40] or [0])
onset=next((t for t in sorted(grms) if t>=2 and grms[t]>max(0.05,base*6)), None)

F,fc_dur,parms = parse_log(fc_bin, ["EV","RCOU","IMU","XKF1","VIBE"])
imu0 = F["IMU"][F["IMU"]["I"]==0].reset_index(drop=True); imu0["w"]=np.linalg.norm(imu0[["GyrX","GyrY","GyrZ"]].values,axis=1)
lag,ncc = align_time(tm, gmag, imu0["t_s"].values, imu0["w"].values)
S["t_fc"] = S["t_rel"] + lag
arm_fc = float(F["EV"][F["EV"]["Id"]==10]["t_s"].max()); disarm_fc = float(F["EV"][F["EV"]["Id"]==11]["t_s"].max())
rc = F["RCOU"]; S["motor_pwm"] = np.interp(S["t_fc"], rc["t_s"], rc["C1"])
S["gyro_mag"] = np.interp(S["t_rel"], tm, gmag)
xkf = F["XKF1"]; xkf = (xkf[xkf["C"]==0] if "C" in xkf else xkf).reset_index(drop=True)
S["ekf_vel"] = np.interp(S["t_fc"], xkf["t_s"].values, np.linalg.norm(xkf[["VN","VE","VD"]].values, axis=1))
vibe = F.get("VIBE")
S["vibe"] = (np.interp(S["t_fc"], vibe["t_s"].values, np.linalg.norm(vibe[["VibeX","VibeY","VibeZ"]].values, axis=1))
             if vibe is not None else np.nan)
def regime(r):
    if r["t_fc"] < arm_fc: return "at_rest"
    if onset is not None and r["t_rel"] < onset: return "motors_on_ground"
    return "in_flight"
S["regime"] = S.apply(regime, axis=1)
print(f"OAK-D<->FC gyro offset: lag {lag:.1f}s (NCC {ncc:.3f}); FC arm {arm_fc:.0f}s disarm {disarm_fc:.0f}s; motion onset (coord) {onset:.0f}s")
print(S["regime"].value_counts().to_dict())

## Sharpness vs. exposure / motion / EKF velocity / vibration

In [ ]:
fig,ax=plt.subplots(2,3,figsize=(18,9))
colors={"at_rest":"seagreen","motors_on_ground":"orange","in_flight":"crimson"}
cc=[colors[r] for r in S.regime]
a=ax[0,0]
for reg,c in colors.items():
    s=S[S.regime==reg]; a.scatter(s.t_rel, s.var_laplacian, c=c, label=reg, s=40)
a2=a.twinx(); a2.plot(S.t_rel, S.exposure_ms, "k.", alpha=.3); a2.set_ylabel("exposure (ms)")
a.set_yscale("log"); a.set_xlabel("coord time (s)"); a.set_ylabel("var(Laplacian) [log]"); a.legend(fontsize=8); a.set_title("sharpness over flight (dots=exposure)")
def sc(ax_, col, xl, ttl):
    ax_.scatter(S[col], S.var_laplacian, c=cc, s=40); ax_.set_xlabel(xl); ax_.set_ylabel("var(Laplacian)")
    ax_.set_yscale("log"); ax_.grid(alpha=.3); ax_.set_title(ttl)
sc(ax[0,1], "exposure_ms", "exposure (ms)", "vs exposure")
sc(ax[0,2], "gyro_mag", "OAK-D gyro (rad/s)", "vs angular rate")
sc(ax[1,0], "ekf_vel", "EKF speed (m/s)", "vs EKF velocity")
sc(ax[1,1], "vibe", "VIBE magnitude", "vs vibration")
a=ax[1,2]; a.boxplot([S[S.regime==r]["var_laplacian"].values for r in colors], labels=list(colors))
a.set_yscale("log"); a.set_ylabel("var(Laplacian)"); a.set_title("distribution by regime"); a.grid(alpha=.3); a.tick_params(axis="x", labelsize=8)
fig.suptitle("Still sharpness vs exposure / motion / EKF velocity / vibration", fontweight="bold")
fig.tight_layout(); fig.savefig(f"{derived}/image-sharpness-vs-motion-260712.png", dpi=110, bbox_inches="tight"); plt.show()

## Sample stills per regime (eyeball)

A handful from each regime, spread across it, as grayscale thumbnails with their metrics -- to visually confirm the numbers.

In [ ]:
n_per=5; picks={}
for reg in ["at_rest","motors_on_ground","in_flight"]:
    s=S[S.regime==reg].sort_values("t_rel")
    if len(s): picks[reg]=s.iloc[np.linspace(0,len(s)-1,min(n_per,len(s))).astype(int)]
rows_=max(1,len(picks))
fig,ax=plt.subplots(rows_, n_per, figsize=(n_per*3.0, rows_*2.7)); ax=np.atleast_2d(ax)
for i,reg in enumerate(picks):
    for j in range(n_per):
        a=ax[i,j]; a.axis("off")
        if j < len(picks[reg]):
            r=picks[reg].iloc[j]
            img=Image.open(f"{cap_dir}/{r['name']}").convert("L"); img.thumbnail((420,420))
            a.imshow(img, cmap="gray")
            a.set_title(f"{reg} t={r.t_rel:.0f}s\nvarLap {r.var_laplacian:.0f} | {r.exposure_ms:.0f}ms | gyro {r.gyro_mag:.2f}", fontsize=7)
fig.suptitle("Sample stills per regime (grayscale)", fontweight="bold")
fig.tight_layout(); fig.savefig(f"{derived}/image-samples-by-regime-260712.png", dpi=110, bbox_inches="tight"); plt.show()

## Answers + conclusion

A frame is **sharp** if `var_laplacian >= the 60th percentile of the at-rest frames` (as sharp as a typical resting frame) -- an explicit, reproducible threshold.

In [ ]:
thr = float(np.percentile(S[S.regime=="at_rest"]["var_laplacian"], 60)) if (S.regime=="at_rest").any() else float(S["var_laplacian"].median())
S["sharp"] = S["var_laplacian"] >= thr
by = S.groupby("regime").agg(n=("name","size"), sharp=("sharp","sum"),
        med_varlap=("var_laplacian","median"), med_expo_ms=("exposure_ms","median"),
        med_gyro=("gyro_mag","median"), med_ekf_vel=("ekf_vel","median"), med_vibe=("vibe","median")).reset_index()
print("sharp threshold (var_laplacian):", round(thr,1)); display(by)
n_rest=int((S.regime=="at_rest").sum()); n_flight=int((S.regime=="in_flight").sum())
sharp_in_flight=int(S[(S.regime=="in_flight") & S.sharp].shape[0])
blur_at_rest=int(S[(S.regime=="at_rest") & ~S.sharp].shape[0])
print(f"\nSHARP while in flight : {sharp_in_flight} / {n_flight}")
print(f"BLURRY while at rest  : {blur_at_rest} / {n_rest}")
def _corr(col):
    x=S[col].astype(float); y=np.log(S.var_laplacian.astype(float)+1); m=np.isfinite(x)&np.isfinite(y)
    return float(np.corrcoef(y[m], x[m])[0,1]) if m.sum()>2 else float("nan")
corr={c:_corr(c) for c in ["exposure_ms","gyro_mag","ekf_vel","vibe"]}
for k,v in corr.items(): print(f"corr(log sharpness, {k:12s}) = {v:+.2f}")

In [ ]:
med={r["regime"]: r for r in by.to_dict(orient="records")}
concl=(f"var(Laplacian) median collapses at-rest->flight: "
       f"{med.get('at_rest',{}).get('med_varlap',float('nan')):.0f} -> {med.get('in_flight',{}).get('med_varlap',float('nan')):.0f}. "
       f"{sharp_in_flight}/{n_flight} in-flight frames reach at-rest sharpness; {blur_at_rest}/{n_rest} at-rest frames are blurry. "
       f"Exposure ~{S.exposure_ms.median():.0f} ms (long); "
       f"corr(sharpness): exposure={corr['exposure_ms']:+.2f} gyro={corr['gyro_mag']:+.2f} "
       f"ekf_vel={corr['ekf_vel']:+.2f} vibe={corr['vibe']:+.2f}.")
print(concl)
summary = {
  "provenance": {"flight": os.path.basename(D), "notebook":"analysis/image-sharpness-vs-motion.ipynb",
     "coordinator_git_sha": git_sha,
     "inputs": {"feat":{"name":os.path.basename(feat),"sha256":sha256(feat)},
                "fc_bin":{"name":os.path.basename(fc_bin),"sha256":sha256(fc_bin)}},
     "n_stills_scored": int(len(S)), "zero_size_in_session": zero,
     "mono_images_saved": False,
     "method":"var-of-Laplacian on grayscale @1024px long side; exposure/ISO from sidecars; OAK-D<->FC offset by gyro cross-correlation; motor/EKF-vel/VIBE interpolated at each still's FC time."},
  "timeline": {"gyro_align_lag_s": round(float(lag),1), "gyro_align_ncc": round(float(ncc),3),
     "fc_arm_s": round(arm_fc,1), "fc_disarm_s": round(disarm_fc,1), "motion_onset_coord_s": round(float(onset),1)},
  "sharp_threshold_var_laplacian": round(thr,1),
  "by_regime": by.to_dict(orient="records"),
  "sharp_in_flight": sharp_in_flight, "blurry_at_rest": blur_at_rest,
  "exposure_ms": {"min": float(S.exposure_ms.min()), "median": float(S.exposure_ms.median()), "max": float(S.exposure_ms.max())},
  "correlations_log_sharpness": {k: round(v,2) for k,v in corr.items()},
  "conclusion": concl,
}
out=f"{derived}/image-sharpness-vs-motion.json"; json.dump(summary, open(out,"w"), indent=2, default=float)
print("wrote", out)